In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

from fairlearn.metrics import MetricFrame, selection_rate, true_positive_rate, false_positive_rate
from fairlearn.reductions import ExponentiatedGradient, DemographicParity
from fairlearn.postprocessing import ThresholdOptimizer

from sklearn.datasets import fetch_openml

In [6]:
# fetch_openml es una función que permite descargar datasets desde la plataforma OpenML.
# Dataset de clasificación binaria para predecir si una persona gana más o menos de $50,000 anuales
adult = fetch_openml(data_id=1590, as_frame=True) # as_frame = True devuelve los datos como un DataFrames de pandas
# adult = fetch_openml(name='adult', version='latest', as_frame=True)

X = adult.data
y = (adult.target == ">50K").astype(int)  # variable binaria ingresos altos

# Atributo sensible: sexo
sensitive_feature = X["sex"]

In [5]:
adult.frame.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,class
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,<=50K


En one-hot encoding podemos encontrarnos con el problema de no saber distinguir entre:
- Un dato faltante (NaN)
- La categoría que se deja fuera por colinealidad (drop_first=True)

Soluciones recomendadas:
1. Tratar los NaN como una catetoria aparte -> Usar dummy_na=True en pd.get_dummies
2. Reemplazar NaN por una categoría como "Unknown" o "Missing"
3. Eliminar filas con NaN si son pocas y no aportan información útil


In [27]:
# Preprocesado simple: eliminar columnas con strings complejos
X = pd.get_dummies(X.drop(columns=["sex", "race"]), drop_first=True)

In [ ]:
print(X.isna().sum().to_string()) #Lista el número de NaN en cada columna

age                                          0
fnlwgt                                       0
education-num                                0
capital-gain                                 0
capital-loss                                 0
hours-per-week                               0
workclass_Local-gov                          0
workclass_Never-worked                       0
workclass_Private                            0
workclass_Self-emp-inc                       0
workclass_Self-emp-not-inc                   0
workclass_State-gov                          0
workclass_Without-pay                        0
education_11th                               0
education_12th                               0
education_1st-4th                            0
education_5th-6th                            0
education_7th-8th                            0
education_9th                                0
education_Assoc-acdm                         0
education_Assoc-voc                          0
education_Bac

In [ ]:
X.columns[X.isna().any()] #Lista las columnas en las que hay algún NaN

Index([], dtype='object')

In [ ]:
X[X.isna().any(axis=1)] #Comprobamos si hay NaN en algun registro

,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week,workclass_Local-gov,workclass_Never-worked,workclass_Private,workclass_Self-emp-inc,...,native-country_Portugal,native-country_Puerto-Rico,native-country_Scotland,native-country_South,native-country_Taiwan,native-country_Thailand,native-country_Trinadad&Tobago,native-country_United-States,native-country_Vietnam,native-country_Yugoslavia


In [28]:
# Split train/test
X_train, X_test, y_train, y_test, sex_train, sex_test = train_test_split(
    X, y, sensitive_feature, test_size=0.3, random_state=42, stratify=y
) #stratify mantiene las proporciones del dataset original dadas las variables de interés

In [ ]:
# Modelo base
scaler = StandardScaler() 
X_train_scaled = scaler.fit_transform(X_train) #Normalizamos datos, LogisticRegression es sensible a órdenes de magnitud
X_test_scaled = scaler.transform(X_test) #Obviamos el fit para evitar data leakage

clf = LogisticRegression(max_iter=200)
clf.fit(X_train_scaled, y_train)

y_pred = clf.predict(X_test_scaled)

print("Accuracy base:", accuracy_score(y_test, y_pred))

Accuracy base: 0.85415955776974


In [ ]:
# Métricas de equidad 
metrics = {
    "accuracy": accuracy_score,         #% de predicciones correctas
    "selection_rate": selection_rate,   #% de personas clasificadas como "ingresos altos"
    "TPR": true_positive_rate,          #% de personas con ingresos altos correctamente identificadas
    "FPR": false_positive_rate,         #% de personas con ingresos bajos incorrectamente clasificadas como "altos"
}

mf = MetricFrame(metrics=metrics, y_true=y_test, y_pred=y_pred, sensitive_features=sex_test)
print("\nMétricas por grupo (modelo base):")
print(mf.by_group)


Métricas por grupo (modelo base):
        accuracy  selection_rate       TPR       FPR
sex                                                 
Female  0.927733        0.079473  0.531646  0.021375
Male    0.817681        0.247346  0.607856  0.091773


In [ ]:
# Mitigación: In-Processing con fairness constraint
constraint = DemographicParity()
mitigator = ExponentiatedGradient(LogisticRegression(max_iter=200), constraint)
mitigator.fit(X_train_scaled, y_train, sensitive_features=sex_train) #Durante el entrenamiento considera tanto la precisión como la equidad

y_pred_fair = mitigator.predict(X_test_scaled)

mf_fair = MetricFrame(metrics=metrics, y_true=y_test, y_pred=y_pred_fair, sensitive_features=sex_test)
print("\nMétricas por grupo (mitigado - InProcessing):")
print(mf_fair.by_group)


Métricas por grupo (mitigado - InProcessing):
        accuracy  selection_rate       TPR       FPR
sex                                                 
Female  0.896026        0.151946  0.710669  0.080158
Male    0.802675        0.157820  0.434473  0.038433


**Trade-off**

Ganancia: Equidad perfecta (selection rate casi igual)

Pérdida: Ligera reducción en accuracy general

In [ ]:
# Mitigación: Post-Processing 
postproc = ThresholdOptimizer(estimator=clf, constraints="demographic_parity",
                             prefit=True)
postproc.fit(X_train_scaled, y_train, sensitive_features=sex_train)

y_pred_post = postproc.predict(X_test_scaled, sensitive_features=sex_test)

mf_post = MetricFrame(metrics=metrics, y_true=y_test, y_pred=y_pred_post, sensitive_features=sex_test)
print("\nMétricas por grupo (mitigado - PostProcessing):")
print(mf_post.by_group)



Métricas por grupo (mitigado - PostProcessing):
        accuracy  selection_rate       TPR       FPR
sex                                                 
Female  0.893556        0.165123  0.757685  0.088987
Male    0.801960        0.155063  0.428717  0.036972
